In [1]:

import torch 
from vit_tiny_main import vit_tiny_classifier
model = vit_tiny_classifier()
# checkpoint = torch.load("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/src/vit_tiny_checkpoints/model_epoch_78.pth")
# model.load_state_dict(checkpoint['model_state_dict'])

loaded


In [ ]:
# features_dict = model.forward_features(image_resized)
#         features = features_dict["x_norm_patchtokens"].view(
#             image_resized.shape[2] // 14, image_resized.shape[3] // 14, -1
#         )

In [7]:
image = torch.rand((1, 3, 224, 224))
model.forward_features(image)['x_norm_patchtokens'].view(
            image.shape[2] // 14, image.shape[3] // 14, -1
        ).shape

torch.Size([16, 16, 384])

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import transforms
import cv2
from tqdm.notebook import tqdm
from icecream import ic
from transformers import get_cosine_schedule_with_warmup
from PIL import Image
import random
import numpy as np
import timm
from extended_kd.vit_tiny_masked import ViT_tiny
from utils import patchify
import cv2
from PIL import Image
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = 'cuda'

In [2]:
root_dir = "/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet/ILSVRC2012_img_val"

class MaskedDataset(Dataset):
    def __init__(self, root_dir = root_dir, split = 'train'):
        super().__init__()
        self.root_dir = root_dir
        self.image_paths = sorted([os.path.join(self.root_dir, p) for p in os.listdir(self.root_dir)])
        
        if split == 'train':
            self.image_paths = self.image_paths[:int(len(self.image_paths)*0.8)]
        else:
            self.image_paths = self.image_paths[int(len(self.image_paths)*0.8):]

        print(self.image_paths[0])    
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = self.image_paths[idx]   # numpy array
        img = Image.open(img).convert("RGB")
        img = self.transform(img)
        return img

train_dataset = MaskedDataset(split='train')
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=8)

test_dataset = MaskedDataset(split='test')
test_dataloader = DataLoader(test_dataset, batch_size=128, num_workers=8)
model = ViT_tiny()

/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet/ILSVRC2012_img_val/ILSVRC2012_val_00000001.JPEG
/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet/ILSVRC2012_img_val/ILSVRC2012_val_00040001.JPEG


In [ ]:
# class MaskedDataset(Dataset):
#     def __init__(self, root_dir = '/media/system/ZERBUIS_EXT_STOR/dynamic_slam/imagenet/ILSVRC2012_img_', split = 'train'):
#         super().__init__()
#         self.root_dir = root_dir + split
#         if split == 'train':  # train has subdirs
#             sub_dirs = [os.path.join(self.root_dir, p) for p in os.listdir(self.root_dir)]
#             self.image_paths = []
#             for s in sub_dirs:
#                 images = [os.path.join(s, p) for p in os.listdir(s)]
#                 self.image_paths.extend(images)
        
#         elif split == 'val': 
#             self.image_paths = [os.path.join(self.root_dir, p) for p in os.listdir(self.root_dir)]
#         else:
#             raise ValueError("Invalid split")

#         self.transform = transforms.Compose([
#             transforms.Resize((224, 224)),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                     std=[0.229, 0.224, 0.225])
#         ])

#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         img = self.image_paths[idx]   # numpy array
#         img = Image.open(img).convert("RGB")
#         img = self.transform(img)
#         return img

# train_dataset = MaskedDataset(split='train')
# train_dataloader = DataLoader(train_dataset, batch_size=212, shuffle=True, num_workers=4)

# test_dataset = MaskedDataset(split='val')
# test_dataloader = DataLoader(test_dataset, batch_size=128)
# model = ViT_tiny()

In [4]:
BATCH_SIZE = 32
LR = 5e-5
EPOCHS = 10
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05

device = "cuda"

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)


In [5]:

for epoch in tqdm(range(10)):
    total_loss = 0
    for images in tqdm(train_dataloader, desc='batches'):

        images = images.cuda()

        pred, mask = model(images)

        target = patchify(images,14)
        pred = pred[:, 1:, :]
        loss = torch.abs(pred - target)

        loss = (loss.mean(dim=-1) * mask).sum() / mask.sum()
        total_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    print(f'epoch loss: {total_loss/len(train_dataloader)}')
    
    eval_loss = 0
    for images in test_dataloader:
        images = images.to(device)
        pred, mask = model(images)

        target = patchify(images,14)
        pred = pred[:, 1:, :]
        loss = torch.abs(pred - target)

        loss = (loss.mean(dim=-1) * mask).sum() / mask.sum()
        eval_loss += loss.item()
    
    print(f'eval loss: {eval_loss/len(test_dataloader)}')

  0%|          | 0/10 [00:00<?, ?it/s]

batches:   0%|          | 0/157 [00:00<?, ?it/s]

epoch loss: 0.9332198603138043
eval loss: 0.7761348025708259


batches:   0%|          | 0/157 [00:00<?, ?it/s]

epoch loss: 0.6946113181721633
eval loss: 0.6047955585431449


...

epoch loss: 0.42557688731296806
eval loss: 0.42336708609061907


batches:   0%|          | 0/157 [00:00<?, ?it/s]

epoch loss: 0.42309275706102895
eval loss: 0.42322270251527616


batches:   0%|          | 0/157 [00:00<?, ?it/s]

epoch loss: 0.42242025114168785
eval loss: 0.42181879470619976
